# train_edge_model

Этот ноутбук обучает edge-модель на уже очищенном датасете из папки `training/data/processed`.

Ожидаемые признаки:

- `Air temperature [K]`
- `temp_diff`
- `Rotational speed [rpm]`
- `Torque [Nm]`
- `power_kw`
- `Tool wear [min]`

Target:

- `Machine failure`


In [9]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


## 1. Пути и конфиг

In [10]:
# Укажи имя своего очищенного файла здесь
DATA_PATH = Path("../data/processed/predictive_maintenance_processed.csv")

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "edge_model.pkl"
FEATURES_PATH = ARTIFACTS_DIR / "edge_feature_list.json"
METRICS_PATH = ARTIFACTS_DIR / "edge_metrics.json"

EDGE_FEATURES = [
    "Air temperature [K]",
    "temp_diff",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "power_kw",
    "Tool wear [min]",
]

TARGET = "Machine failure"


## 2. Загрузка очищенного датасета

In [11]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (10000, 8)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,temp_diff,power_kw
0,298.1,308.6,1551,42.8,0,0,10.5,6.951591
1,298.2,308.7,1408,46.3,3,0,10.5,6.826723
2,298.1,308.5,1498,49.4,5,0,10.4,7.749388
3,298.2,308.6,1433,39.5,7,0,10.4,5.927505
4,298.2,308.7,1408,40.0,9,0,10.5,5.897817


## 3. Проверка нужных колонок

In [12]:
required_columns = EDGE_FEATURES + [TARGET]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")
print(df[required_columns].dtypes)


All required columns are present.
Air temperature [K]       float64
temp_diff                 float64
Rotational speed [rpm]      int64
Torque [Nm]               float64
power_kw                  float64
Tool wear [min]             int64
Machine failure             int64
dtype: object


## 4. Формируем X и y

In [13]:
X = df[EDGE_FEATURES].copy()
y = df[TARGET].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())
X.head()


X shape: (10000, 6)
y shape: (10000,)

Class distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
0,298.1,10.5,1551,42.8,6.951591,0
1,298.2,10.5,1408,46.3,6.826723,3
2,298.1,10.4,1498,49.4,7.749388,5
3,298.2,10.4,1433,39.5,5.927505,7
4,298.2,10.5,1408,40.0,5.897817,9


## 5. Делим на train / test

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)


Train: (8000, 6) (8000,)
Test : (2000, 6) (2000,)


## 6. Обучение edge-модели

Используем `RandomForestClassifier` как лёгкую модель для edge-слоя.


In [15]:
model = RandomForestClassifier(
    n_estimators=120,
    max_depth=10,
    min_samples_split=8,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Model trained successfully")


Model trained successfully


## 7. Оценка модели

In [16]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "model_type": "edge",
    "model_name": "RandomForestClassifier",
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 6),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 6),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 6),
    "f1_score": round(float(f1_score(y_test, y_pred, zero_division=0)), 6),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 6),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "features": EDGE_FEATURES,
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
}

print(classification_report(y_test, y_pred, digits=4))
print(json.dumps(metrics, indent=2))


              precision    recall  f1-score   support

           0     0.9948    0.9917    0.9933      1932
           1     0.7838    0.8529    0.8169        68

    accuracy                         0.9870      2000
   macro avg     0.8893    0.9223    0.9051      2000
weighted avg     0.9876    0.9870    0.9873      2000

{
  "model_type": "edge",
  "model_name": "RandomForestClassifier",
  "accuracy": 0.987,
  "precision": 0.783784,
  "recall": 0.852941,
  "f1_score": 0.816901,
  "roc_auc": 0.975673,
  "confusion_matrix": [
    [
      1916,
      16
    ],
    [
      10,
      58
    ]
  ],
  "features": [
    "Air temperature [K]",
    "temp_diff",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "power_kw",
    "Tool wear [min]"
  ],
  "train_size": 8000,
  "test_size": 2000
}


## 8. Важность признаков

In [17]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance


,feature,importance
3,Torque [Nm],0.229862
5,Tool wear [min],0.217743
2,Rotational speed [rpm],0.209915
4,power_kw,0.200462
1,temp_diff,0.099416
0,Air temperature [K],0.042602


## 9. Сохранение модели и артефактов

In [18]:
joblib.dump(model, MODEL_PATH)

with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(EDGE_FEATURES, f, ensure_ascii=False, indent=2)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved feature list to:", FEATURES_PATH)
print("Saved metrics to:", METRICS_PATH)


Saved model to: ..\artifacts\edge_model.pkl
Saved feature list to: ..\artifacts\edge_feature_list.json
Saved metrics to: ..\artifacts\edge_metrics.json


## 10. Пример одного предсказания

In [19]:
sample_input = X.iloc[[0]].copy()
sample_risk = float(model.predict_proba(sample_input)[0, 1])
sample_pred = int(model.predict(sample_input)[0])

result = {
    "risk_score": round(sample_risk, 4),
    "prediction": "HIGH_RISK" if sample_risk >= 0.5 else "LOW_RISK",
    "features": sample_input.to_dict(orient="records")[0],
}

print(json.dumps(result, indent=2))


{
  "risk_score": 0.0007,
  "prediction": "LOW_RISK",
  "features": {
    "Air temperature [K]": 298.1,
    "temp_diff": 10.5,
    "Rotational speed [rpm]": 1551,
    "Torque [Nm]": 42.8,
    "power_kw": 6.95159056015735,
    "Tool wear [min]": 0
  }
}
